# tm_final_33 - Financial Tweet Sentiment Classification
## Text Mining 2025/2026 - NOVA IMS
### Clean Pipeline - Single Best Model - Restart & Run All

**Model:** Twitter-RoBERTa-large topic-sentiment (`cardiffnlp/twitter-roberta-large-topic-sentiment-latest`), fine-tuned end-to-end (all ~355M parameters) for 3-class financial sentiment. This large, Twitter-domain model was selected after an expanded Hugging Face backbone search in `tm_tests_33.ipynb`; it gives the best out-of-fold macro-F1.

**Recipe:** 5-fold stratified CV. Each fold fine-tunes the full network for 4 epochs with a class-weighted loss (to counter the 4.3:1 imbalance), bf16 mixed precision on GPU. We report the out-of-fold (OOF) F1-macro - an honest, leak-free estimate - and average the five folds' test probabilities for the final prediction (a robust CV-ensemble of one architecture).

This is a **single-model** pipeline: one model family, one training recipe. It is also our best overall result, surpassing the previous RoBERTa-large-2022 candidate and the heterogeneous 6-learner stacking ensemble, which lives with every other experiment in `tm_tests_33.ipynb`.

**Expected OOF F1-macro:** approx 0.8873  (baseline LightGBM+RoBERTa: 0.8021)

**Runtime:** ~20 min on a CUDA GPU (e.g. RTX 5070, batch 16, bf16); much slower on CPU (falls back to lighter partial fine-tuning automatically). **Instructions:** Kernel -> Restart Kernel and Run All Cells.


In [1]:
# Cell 1: Imports and reproducibility
import os, sys, time, gc, warnings
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                             recall_score, classification_report)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

SEED = 42

def seed_all(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all()
print(f'Seed fixed: {SEED}')
print(f'Python: {sys.version[:20]} | torch {torch.__version__}')


Seed fixed: 42
Python: 3.11.9 (tags/v3.11.9 | torch 2.12.0+cu130


In [2]:
# Cell 2: Load train and test datasets
train = pd.read_csv('data/raw/train.csv')
test  = pd.read_csv('data/raw/test.csv')

texts      = train['text'].tolist()
test_texts = test['text'].tolist()
y = train['label'].values

print(f'Train: {train.shape} | Test: {test.shape}')
print('Label distribution (0=Bearish, 1=Bullish, 2=Neutral):')
print(train['label'].value_counts().sort_index())


Train: (9543, 2) | Test: (2388, 2)
Label distribution (0=Bearish, 1=Bullish, 2=Neutral):
label
0    1442
1    1923
2    6178
Name: count, dtype: int64


In [3]:
# Cell 3: Preprocessing
# Best config from ablation: RAW text (no preprocessing). Financial ticker symbols
# ($AAPL, $TSLA), cashtags, analyst names and domain terms carry strong predictive
# signal that aggressive cleaning would destroy. The Twitter-RoBERTa tokenizer was
# pre-trained on tweets, so it already handles @mentions, hashtags and URLs natively.
print('Using raw text — no preprocessing.')
print(f'Sample: {texts[0][:100]}')


Using raw text — no preprocessing.
Sample: $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT


In [4]:
# Cell 4: Configuration and device selection
MODEL_NAME   = 'cardiffnlp/twitter-roberta-large-topic-sentiment-latest'
MAXLEN       = 96       # tweets are short; 96 tokens covers >99% with headroom
LR           = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS       = 4
N_FOLDS      = 5

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    BATCH, EVAL_BATCH, FULL_FT = 16, 32, True
    AMP_DTYPE = torch.bfloat16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name(0)} | '
          f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB | '
          f'FULL fine-tuning, batch {BATCH}, bf16')
else:
    BATCH, EVAL_BATCH, FULL_FT = 8, 32, False
    AMP_DTYPE = None
    torch.set_num_threads(min(12, os.cpu_count()))
    print(f'CPU fallback: partial fine-tuning (top 4 layers), batch {BATCH}')

TRAIN_TOP_LAYERS = 4  # used only on the CPU fallback

# Inverse-frequency class weights -> counter the 4.3:1 imbalance (Neutral dominates)
counts = np.bincount(y, minlength=3)
class_weights = torch.tensor(len(y) / (3 * counts), dtype=torch.float32)
print(f'class counts={counts.tolist()} weights={[round(x,3) for x in class_weights.tolist()]}')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)


GPU: NVIDIA GeForce RTX 5070 | VRAM 12.8 GB | FULL fine-tuning, batch 16, bf16
class counts=[1442, 1923, 6178] weights=[2.206, 1.654, 0.515]


In [5]:
# Cell 5: Model builder, training loop and inference helpers

def build_model():
    """Load Twitter-RoBERTa with a fresh 3-class head. On GPU every layer is
    trainable (full fine-tuning); on CPU we freeze all but the top 4 encoder layers."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True)
    if not FULL_FT:
        for p in model.parameters():
            p.requires_grad = False
        unfreeze_from = model.config.num_hidden_layers - TRAIN_TOP_LAYERS
        for name, p in model.named_parameters():
            if 'classifier' in name:
                p.requires_grad = True
            elif 'encoder.layer.' in name:
                li = int(name.split('encoder.layer.')[1].split('.')[0])
                if li >= unfreeze_from:
                    p.requires_grad = True
    return model.to(DEVICE)


def train_fold(tr_texts, tr_labels, fold):
    """Fine-tune one fold: AdamW + linear warmup (10%), grad-clip 1.0,
    class-weighted cross-entropy, bf16 autocast on GPU."""
    seed_all(SEED + fold)
    model = build_model(); model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = int(np.ceil(len(tr_texts) / BATCH))
    total = steps_per_epoch * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    n = len(tr_texts)
    for epoch in range(EPOCHS):
        order = np.random.permutation(n)
        running, t0 = 0.0, time.time()
        for i in range(0, n, BATCH):
            bidx = order[i:i + BATCH]
            bt = [tr_texts[j] for j in bidx]
            bl = torch.tensor([tr_labels[j] for j in bidx], dtype=torch.long, device=DEVICE)
            enc = tok(bt, padding=True, truncation=True, max_length=MAXLEN, return_tensors='pt')
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            opt.zero_grad()
            if DEVICE == 'cuda':
                with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                    loss = loss_fn(model(**enc).logits, bl)
            else:
                loss = loss_fn(model(**enc).logits, bl)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step()
            running += loss.item()
        print(f'    fold {fold} epoch {epoch+1}/{EPOCHS} loss={running/steps_per_epoch:.4f} '
              f'({time.time()-t0:.0f}s)')
    return model


@torch.no_grad()
def predict_proba(model, txts):
    model.eval()
    out = []
    for i in range(0, len(txts), EVAL_BATCH):
        enc = tok(txts[i:i + EVAL_BATCH], padding=True, truncation=True,
                  max_length=MAXLEN, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        if DEVICE == 'cuda':
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                logits = model(**enc).logits
        else:
            logits = model(**enc).logits
        out.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(out)

print('Helpers defined.')


Helpers defined.


In [6]:
# Cell 6: 5-fold CV fine-tuning -> OOF F1-macro + averaged test probabilities
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_proba      = np.zeros((len(y), 3), dtype=np.float32)
test_proba_acc = np.zeros((len(test_texts), 3), dtype=np.float32)
fold_f1 = []
t_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(cv.split(texts, y)):
    print(f'\n=== Fold {fold+1}/{N_FOLDS} ===')
    tr_texts  = [texts[i] for i in tr_idx]
    tr_labels = [int(y[i]) for i in tr_idx]
    va_texts  = [texts[i] for i in va_idx]

    model = train_fold(tr_texts, tr_labels, fold)

    va_proba = predict_proba(model, va_texts)
    oof_proba[va_idx] = va_proba
    f1 = f1_score(y[va_idx], va_proba.argmax(1), average='macro')
    fold_f1.append(f1)
    print(f'  fold {fold+1} val F1-macro = {f1:.4f}')

    test_proba_acc += predict_proba(model, test_texts)   # accumulate for averaging
    del model; gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

test_proba = test_proba_acc / N_FOLDS
print(f'\nDone in {(time.time()-t_start)/60:.1f} min.')


=== Fold 1/5 ===
  fold 1 val F1-macro = 0.8915

=== Fold 2/5 ===
  fold 2 val F1-macro = 0.8791

=== Fold 3/5 ===
  fold 3 val F1-macro = 0.8845

=== Fold 4/5 ===
  fold 4 val F1-macro = 0.8920

=== Fold 5/5 ===
  fold 5 val F1-macro = 0.8897

Done in 20.1 min.


In [7]:
# Cell 7: Out-of-fold performance (honest, leak-free estimate)
oof_pred = oof_proba.argmax(1)
oof_f1   = f1_score(y, oof_pred, average='macro')

print(f'OOF F1-macro : {oof_f1:.4f}  (per-fold {[round(x,4) for x in fold_f1]}, '
      f'std {np.std(fold_f1):.4f})')
print(f'Accuracy     : {accuracy_score(y, oof_pred):.4f}')
print(f'Precision-mac: {precision_score(y, oof_pred, average="macro"):.4f}')
print(f'Recall-macro : {recall_score(y, oof_pred, average="macro"):.4f}')
print(f'Baseline     : 0.8021 (LightGBM+RoBERTa)  ->  improvement {oof_f1-0.8021:+.4f}')
print()
print(classification_report(y, oof_pred, target_names=['Bearish', 'Bullish', 'Neutral']))


OOF F1-macro : 0.8873  (per-fold [0.8915, 0.8791, 0.8845, 0.892, 0.8897], std 0.0049)
Accuracy     : 0.9090
Precision-mac: 0.8796
Recall-macro : 0.8958
Baseline     : 0.8021 (LightGBM+RoBERTa)  ->  improvement +0.0852

              precision    recall  f1-score   support

     Bearish       0.84      0.86      0.85      1442
     Bullish       0.85      0.90      0.88      1923
     Neutral       0.94      0.92      0.93      6178

    accuracy                           0.91      9543
   macro avg       0.88      0.90      0.89      9543
weighted avg       0.91      0.91      0.91      9543


In [8]:
# Cell 8: Generate Predictions and Save pred_33.csv
test_pred  = test_proba.argmax(1)
submission = pd.DataFrame({'id': test['id'], 'label': test_pred})
submission.to_csv('pred_33.csv', index=False)
os.makedirs('results/predictions', exist_ok=True)
submission.to_csv('results/predictions/pred_final.csv', index=False)

print(f'Predictions saved: pred_33.csv ({len(submission)} rows)')
print('Distribution:')
print(submission['label'].value_counts().sort_index()
      .rename({0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}))
print()

assert len(submission) == len(test), f'Expected {len(test)}, got {len(submission)}'
assert submission['label'].nunique() == 3, 'Model predicts fewer than 3 classes!'
assert list(submission.columns) == ['id', 'label'], 'Columns must be exactly [id, label]'
print('All assertions PASSED.')
print()
print(submission.head(10).to_string(index=False))


Predictions saved: pred_33.csv (2388 rows)
Distribution:
label
Bearish     361
Bullish     512
Neutral    1515
Name: count, dtype: int64

All assertions PASSED.

 id  label
  0      1
  1      2
  2      2
  3      1
  4      2
  5      1
  6      0
  7      0
  8      2
  9      2
